In [1]:
import torch

In [2]:
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [3]:
x.requires_grad_(True)
x.grad # None by default

In [4]:
y = 2*torch.dot(x,x)
y

tensor(28., grad_fn=<MulBackward0>)

In [5]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

Gradient of $\bold{y}=2\bold{x}^T\bold{x}$ should be $4\bold{x}$

In [6]:
x.grad == 4*x

tensor([True, True, True, True])

In [7]:
x.grad.zero_()
y=x.sum()
y.backward() # Gradient of y WRT x
x.grad # above gradient 

tensor([1., 1., 1., 1.])

In [8]:
x.grad.zero_()
y=x*x
u=y.detach()
z=u*x

z.sum().backward()
x.grad == u # u is a matrix, x is a vector. u is symmetric since u=x*x, so u=u^T. z=u*x so grad_x z = u^T = u

tensor([True, True, True, True])

In [9]:
x.grad.zero_()
y.sum().backward()
x.grad == 2*x

tensor([True, True, True, True])

In [10]:
def f(a):
    # Defines a piecewise fn to demonstrate how PyTorch can calculate gradients no matter the control flow
    b=a*2
    while b.norm() < 1000:
        b=b*2
    if b.sum() > 0:
        c = b
    else:
        c = 100*b
    return c

In [11]:
a=torch.randn(size=(), requires_grad=True)
d=f(a)
d.backward()

a.grad, d/a # as of 7/8/2025, torch.Tensor.__eq__() returns False if one tensor has a gradient fn and another does not

(tensor(512.), tensor(512., grad_fn=<DivBackward0>))

1. Why is the second derivative much more expensive to compute than the first derivative?

The second derivative is the derivative of the first derivative function, requiring the function of the first derivative to first be calculated before the second derivative can be calculated.

2. After running the function for backpropagation, immediately run it again and see what happens. Investigate.

In [12]:
x = torch.arange(4.0)
x.requires_grad_(True)
y=torch.dot(x,x)
y.backward()

y

tensor(14., grad_fn=<DotBackward0>)

In [13]:
x = torch.arange(4.0)
x.requires_grad_(True)
y=torch.dot(x,x)
y.backward(retain_graph=True)
y.backward(retain_graph=True)

y

tensor(14., grad_fn=<DotBackward0>)

Nothing happens when the backward function is called twice as long as the graph is retained. If the graph is not retained, the first call will clear the graph, and the second call will have no graph to calculate the gradient.

3. In the control flow example where we calculate the derivative of $d$ with respect to $a$, what would happen if we changed the variable $a$ to a random vector or a matrix? At this point, the result of the calculation $f(a)$ is no longer a scalar. What happens to the result? How do we analyze this?

In [ ]:
a=torch.arange(12.0)
a=torch.arange(12.0).reshape(4,3)
a.requires_grad_(True)
d=f(a) 

d

tensor([  0.,  64., 128., 192., 256., 320., 384., 448., 512., 576., 640., 704.],
       grad_fn=<MulBackward0>)

If we change the variable $a$ to a random vector or matrix, the output becomes a vector or matrix, respectively. We can analyze this by feeding it into a scalar-valued function and applying the derivative to its scalar output.

4. Let $f(x)=\sin(x)$. Plot the graph of $f$ and its derivative $f'$. Do not exploit the fact that $f'(x)=\cos(x)$ but rather use automatic differentiation to get the reuslt.

5. Let $f(x)=((\log x^2) \cdot \sin x)+x^{-1}$. Write out a dependency graph tracing results from $x$ to $f(x)$.

6. Use the chain rule to compute the derivative $\frac{df}{dx}$ of the aforementioned function, placing each term on the dependency graph that you constructed previously.

7. Given the graph and the intermediate derivative results, you have a number of options when computing the gradient. Evaluate the resul once starting from $x$ to $f$ and once from $f$ tracing back to $x$. The path from $x$ to $f$ is commonly known as forward differentiation, whereas the path from $f$ to $x$ is known as backward differentiation.

8. When might you want to use forward, and when backward, differentiation? Hint: consider the amount of intermediate data needed, the ability to parallelize steps, and the size of matrices and vectors involved.